In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("aula02_lineage_dag")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 13:04:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv("/workspace/datasets/02_lineage_DAG/clientes.csv", header= True, inferSchema= True)

In [4]:
df2 = (
    df.filter(F.col("idade") > 30)
    .select("nome", "idade", "cidade")
    .withColumn("idade_futura", F.col("idade") + 1)
)

In [5]:
df_a = (
    df
    .filter(F.col("idade") > 30)
    .filter(F.col("ativo") == True)
    .filter(F.col("renda_mensal") > 5000)
)

In [6]:
df_b = (
    df.filter(
        (F.col("idade") > 30) &
        (F.col("ativo") == True) &
        (F.col("renda_mensal") > 5000)
    )
)

In [7]:
df_a.explain(True)

== Parsed Logical Plan ==
'Filter ('renda_mensal > 5000)
+- Filter (ativo#22 = true)
   +- Filter (idade#19 > 30)
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, renda_mensal: double, ativo: boolean
Filter (renda_mensal#21 > cast(5000 as double))
+- Filter (ativo#22 = true)
   +- Filter (idade#19 > 30)
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Optimized Logical Plan ==
Filter (((isnotnull(idade#19) AND isnotnull(ativo#22)) AND isnotnull(renda_mensal#21)) AND ((idade#19 > 30) AND (ativo#22 AND (renda_mensal#21 > 5000.0))))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Physical Plan ==
*(1) Filter (((((isnotnull(idade#19) AND isnotnull(ativo#22)) AND isnotnull(renda_mensal#21)) AND (idade#19 > 30)) AND ativo#22) AND (renda_mensal#21 > 5000.0))
+- FileScan csv [cliente

In [8]:
df_b.explain(True)

== Parsed Logical Plan ==
'Filter ((('idade > 30) AND ('ativo = true)) AND ('renda_mensal > 5000))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, renda_mensal: double, ativo: boolean
Filter (((idade#19 > 30) AND (ativo#22 = true)) AND (renda_mensal#21 > cast(5000 as double)))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Optimized Logical Plan ==
Filter (((isnotnull(idade#19) AND isnotnull(ativo#22)) AND isnotnull(renda_mensal#21)) AND (((idade#19 > 30) AND ativo#22) AND (renda_mensal#21 > 5000.0)))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Physical Plan ==
*(1) Filter (((((isnotnull(idade#19) AND isnotnull(ativo#22)) AND isnotnull(renda_mensal#21)) AND (idade#19 > 30)) AND ativo#22) AND (renda_mensal#21 > 5000.0))
+- FileScan csv [cliente_id#17,nome#18,idade#19,cidade#20,rend

In [6]:
print(df2)

DataFrame[nome: string, idade: int, cidade: string, idade_futura: int]


In [7]:
df2.explain(True)

== Parsed Logical Plan ==
'Project [nome#18, idade#19, cidade#20, ('idade + 1) AS idade_futura#33]
+- Project [nome#18, idade#19, cidade#20]
   +- Filter (idade#19 > 30)
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Analyzed Logical Plan ==
nome: string, idade: int, cidade: string, idade_futura: int
Project [nome#18, idade#19, cidade#20, (idade#19 + 1) AS idade_futura#33]
+- Project [nome#18, idade#19, cidade#20]
   +- Filter (idade#19 > 30)
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Optimized Logical Plan ==
Project [nome#18, idade#19, cidade#20, (idade#19 + 1) AS idade_futura#33]
+- Filter (isnotnull(idade#19) AND (idade#19 > 30))
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Physical Plan ==
*(1) Project [nome#18, idade#19, cidade#20, (idade#19 + 1) AS idade_futura#33]
+- *(1) Filter (isnotnull(idade#19) AND (idade#19 > 30))
   +- FileScan c

In [8]:
spark.stop()